# Donor upgrade scoring — explanatory OLS on gift size


## 1. Business Understanding

**EXPLANATORY (regression).** We estimate **associations** between supporter attributes and **average gift** (`avg_gift_size`) to inform major-gift prospecting — **not** individual-level causal effects.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'donor_master.csv')
df['log_gift'] = np.log1p(df['avg_gift_size'].clip(lower=0))
TARGET = 'log_gift'
df[['total_lifetime_value','donation_frequency','num_campaigns','avg_gift_size']].describe()


In [ ]:
df['acquisition_channel'] = df['acquisition_channel'].fillna('Unknown')
df['supporter_type'] = df['supporter_type'].fillna('Unknown')


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num = ['total_lifetime_value','donation_frequency','num_campaigns','days_since_last_donation','is_recurring_donor']
cat = ['acquisition_channel','supporter_type']
m = df[num + cat + [TARGET]].dropna()
X_train, X_test, y_train, y_test = train_test_split(m[num+cat], m[TARGET], test_size=0.2, random_state=42)
prep = ColumnTransformer([
 ('n', Pipeline([('im',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num),
 ('c', Pipeline([('im',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)])


## 3. Modeling & Feature Selection

OLS via statsmodels on full dummy matrix (train only for evaluation split).


In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, r2_score
Xe = pd.get_dummies(X_train, drop_first=True).apply(pd.to_numeric, errors='coerce').fillna(0.0)
Xe = sm.add_constant(Xe, has_constant='add')
model = sm.OLS(np.asarray(y_train, dtype=float), np.asarray(Xe, dtype=float)).fit()
print(model.summary())


## 4. Evaluation & Interpretation


In [ ]:
Xt = pd.get_dummies(X_test, drop_first=True)
Xt = Xt.reindex(columns=[c for c in Xe.columns if c != 'const'], fill_value=0)
Xt = sm.add_constant(Xt, has_constant='add')
Xt = Xt[Xe.columns]
pred = model.predict(Xt)
print('MAE log scale', mean_absolute_error(y_test, pred))
print('Test R2', r2_score(y_test, pred))


## 5. Causal / Relationship Analysis

Coefficients are **associational**; unobserved wealth and engagement channels confound simple OLS.


## 6. Deployment Notes

**No .sav artifact (lighter pipeline).** Deployment would wrap the fitted `statsmodels` equation in the API or re-implement coefficients; for production, refit `sklearn` Pipeline for serialization consistency.
